# SARSA (State-Action-Reward-State-Action)

**SARSA (State-Action-Reward-State-Action)** adalah algoritma Reinforcement Learning berbasis *TD-Learning (Temporal Difference)* yang sangat mirip dengan Q-Learning, tetapi dengan satu perbedaan mendasar:

**SARSA bersifat *On-Policy*, sedangkan Q-Learning bersifat *Off-Policy*.**

## Perbedaan Utama: SARSA vs Q-Learning

* **Q-Learning (*Off-Policy*):** Mengasumsikan agen selalu mengambil aksi **terbaik** `max Q` di state berikutnya, tidak peduli apa aksi yang sebenarnya dipilih oleh kebijakan (`ε-greedy`). Sikapnya lebih *agresif/optimistis*.
* **SARSA (*On-Policy*):** Menggunakan aksi `a'` yang **benar-benar dipilih dan dilakukan** oleh kebijakan (`ε-greedy`) di langkah berikutnya. Sikapnya lebih *konservatif/hati-hati*.

```text
Q-Learning : (s, a, r, s')     → Memilih max Q(s', a') tanpa peduli eksplorasi
SARSA      : (s, a, r, s', a') → Memperhitungkan aksi a' yang riil dijalankan
```

## Rumus Pembaruan SARSA

$$
Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma Q(s', a') - Q(s, a) \right]
$$

## Studi Kasus Analogi: Tebing/Jurang (Cliff Walking)

Bayangkan agen berjalan di pinggir jurang untuk mencapai tujuan:

* **Q-Learning** akan memilih jalur paling pendek persis di tepi jurang karena mengasumsikan ia tidak pernah salah langkah. Hasilnya: jalur optimal, tetapi risiko jatuh tinggi selama latihan jika masih ada eksplorasi acak.
* **SARSA** memperhitungkan fakta bahwa ia kadang melakukan langkah acak (eksplorasi). Karena itu, SARSA memilih jalur memutar yang lebih jauh tapi **lebih aman** dari tepi jurang.


In [5]:
import numpy as np

# Setup dasar
num_states = 9
num_actions = 4
q_table = np.zeros((num_states, num_actions))

alpha = 0.1
gamma = 0.9
epsilon = 0.2

def choose_action(state):
    # Epsilon-greedy action selection
    if np.random.rand() < epsilon:
        return np.random.randint(num_actions)
    else:
        return np.argmax(q_table[state])

def get_next_state_and_reward(state, action):
    # Mapping pergerakan grid 3x3
    row, col = divmod(state, 3)

    if action == 0 and row > 0: row -= 1    # Up
    elif action == 1 and row < 2: row += 1  # Down
    elif action == 2 and col > 0: col -= 1  # Left
    elif action == 3 and col < 2: col += 1  # Right

    next_state = row * 3 + col

    # Posisi khusus: HAZARD=2, WALL=4, GOAL=8
    if next_state == 4:
        return state, -10, False  # Nabrak dinding, posisi tidak berubah
    elif next_state == 2:
        return next_state, -50, True  # Masuk bahaya, episode selesai
    elif next_state == 8:
        return next_state, 100, True  # Berhasil mencapai goal, episode selesai
    else:
        return next_state, -1, False  # Langkah biasa

# Training Loop SARSA
for episode in range(1000):
    state = 0  # START
    action = choose_action(state)  # 1. Pilih Aksi Pertama (a)
    done = False

    while not done:
        # 2. Jalankan aksi, dapatkan reward (r) dan state baru (s')
        next_state, reward, done = get_next_state_and_reward(state, action)

        # 3. Pilih aksi berikutnya (a') SEBELUM update Q-table
        next_action = choose_action(next_state)

        # 4. Update SARSA dengan Q(s', a') - BUKAN max Q(s', a')
        q_table[state, action] += alpha * (
            reward + gamma * q_table[next_state, next_action] - q_table[state, action]
        )

        # 5. Lanjut ke step berikutnya
        state = next_state
        action = next_action

print("Training SARSA Selesai.\n")

# 1. Cetak Q-Table
print("Q-Table Hasil Training (dibulatkan 1 desimal):")
print(np.round(q_table, 1))

# 2. Cetak Arah Gerakan Terbaik per State (Policy)
actions_map = {0: "↑", 1: "↓", 2: "←", 3: "→"}
best_policy = [actions_map[np.argmax(q_table[s])] for s in range(num_states)]

# Tampilkan dalam bentuk Grid 3x3
grid_policy = np.array(best_policy).reshape(3, 3)
grid_policy[0, 2] = "X"  # Hazard
grid_policy[1, 1] = "W"  # Wall
grid_policy[2, 2] = "G"  # Goal

print("\nGrid Visualisasi Langkah Terbaik (Policy):")
print(grid_policy)

Training SARSA Selesai.

Q-Table Hasil Training (dibulatkan 1 desimal):
[[ 53.6  65.   52.5  38.8]
 [  3.   -2.8  52.4 -20.5]
 [  0.    0.    0.    0. ]
 [ 51.1  74.3  64.8  53.5]
 [  0.    0.    0.    0. ]
 [  0.    0.    0.    0. ]
 [ 62.5  73.6  76.2  86.2]
 [ 77.   86.8  73.3 100. ]
 [  0.    0.    0.    0. ]]

Grid Visualisasi Langkah Terbaik (Policy):
[['↓' '←' 'X']
 ['↓' 'W' '↑']
 ['→' '→' 'G']]
